In [1]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import os
from PIL import Image
import yaml
import cv2
from collections import OrderedDict
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

# 📌 Inicialização
print("🔧 Inicializando dispositivo...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo utilizado: {device}")

# 📂 Configuração YAML
print("📂 Carregando configuração YAML...")
with open("dataset.yaml", "r") as file:
    dataset_config = yaml.safe_load(file)

train_img_dir = dataset_config["train"]
val_img_dir = dataset_config["val"]
test_img_dir = dataset_config["test"]
nc = dataset_config["nc"]
class_names = dataset_config["names"]
print(f"✅ Dataset carregado: {len(class_names)} classes")

# 📦 Funções de leitura de labels
def load_yolo_labels(label_path, img_width, img_height):
    boxes, labels = [], []
    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, xc, yc, w, h = map(float, line.strip().split())
            x_min = (xc - w / 2) * img_width
            y_min = (yc - h / 2) * img_height
            x_max = (xc + w / 2) * img_width
            y_max = (yc + h / 2) * img_height
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(int(cls))
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

def replace_extension(file):
    return os.path.splitext(file)[0] + ".txt"

# 📚 Dataset personalizado
class YoloDataset(Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = img_dir.replace("images", "labels")
        self.transforms = transforms
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(('jpg', 'png', 'jpeg'))]

        filtered_imgs = []
        for img in self.imgs:
            label_path = os.path.join(self.label_dir, replace_extension(img))
            if os.path.exists(label_path):
                with open(label_path, 'r') as file:
                    if file.read().strip():
                        filtered_imgs.append(img)
        self.imgs = filtered_imgs
        print(f"📚 Dataset inicializado: {img_dir} com {len(self.imgs)} imagens válidas")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        label_path = os.path.join(self.label_dir, replace_extension(self.imgs[idx]))

        img = Image.open(img_path).convert("RGB")
        img_width, img_height = img.size
        boxes, labels = load_yolo_labels(label_path, img_width, img_height)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}

        if self.transforms:
            img = self.transforms(img)

        return img, target

# 🔁 Transforms
transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor()
])

# 📦 Loaders
train_dataset = YoloDataset(train_img_dir, transform)
val_dataset = YoloDataset(val_img_dir, transform)
test_dataset = YoloDataset(test_img_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# 🧠 Backbone com blocos residuais simples
class ResidualLikeCNNBackbone(nn.Module):
    def __init__(self):
        super(ResidualLikeCNNBackbone, self).__init__()

        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels)
            )

        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.block1 = conv_block(32, 64)
        self.shortcut1 = nn.Conv2d(32, 64, kernel_size=1)
        self.block2 = conv_block(64, 128)
        self.shortcut2 = nn.Conv2d(64, 128, kernel_size=1)
        self.pool = nn.MaxPool2d(2)
        self.out_channels = 128

    def forward(self, x):
        x = self.layer1(x)
        res1 = self.block1(x) + self.shortcut1(x)
        res1 = F.relu(res1)
        res1 = self.pool(res1)

        res2 = self.block2(res1) + self.shortcut2(res1)
        res2 = F.relu(res2)
        res2 = self.pool(res2)

        return OrderedDict([("0", res2)])

# 🛠️ Montagem do modelo
print("🔨 Construindo modelo...")
backbone = ResidualLikeCNNBackbone()
anchor_generator = AnchorGenerator(sizes=((32, 64, 128, 256, 512),), aspect_ratios=((0.5, 1.0, 2.0),))
roi_pooler = MultiScaleRoIAlign(featmap_names=['0'], output_size=7, sampling_ratio=2)

model = torchvision.models.detection.FasterRCNN(
    backbone, num_classes=nc + 1,
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=roi_pooler
)
model.to(device)
print("✅ Modelo pronto para treino")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 🚀 Treinamento
print("🚀 Iniciando treinamento...")
model.train()
prev_loss = float('inf')
patience = 2
no_improve_epochs = 0
max_epochs = 6

for epoch in range(max_epochs):
    total_loss = 0
    for batch_idx, (imgs, targets) in enumerate(train_loader, 1):
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        print(f"📈 Epoch {epoch+1}, Batch {batch_idx}/{len(train_loader)}, Batch loss: {losses.item():.4f}")

    print(f"🎯 Epoch {epoch+1} completa. Loss Total: {total_loss:.4f}")

    if total_loss + 0.1 < prev_loss:
        print("✅ Melhorando, continuando...")
        prev_loss = total_loss
        no_improve_epochs = 0
        torch.save(model.state_dict(), "models/ResidualLikeCNNBackbone_yolo_label.pth")
        print("💾 Modelo salvo!")
    else:
        no_improve_epochs += 1
        print("⚠️ Não houve melhoria.")
        if no_improve_epochs >= patience:
            print("🛑 Parando o treinamento por falta de melhoria.")
            break


🔧 Inicializando dispositivo...
✅ Dispositivo utilizado: cuda
📂 Carregando configuração YAML...
✅ Dataset carregado: 9 classes
📚 Dataset inicializado: C:\\Users\\Jorge Cunha\\Desktop\\Python\\AI_Detection_Cars\\AAU2\\DetectObjects_JC\\dataset\\train\\images com 1258 imagens válidas
📚 Dataset inicializado: C:\\Users\\Jorge Cunha\\Desktop\\Python\\AI_Detection_Cars\\AAU2\\DetectObjects_JC\\dataset\\val\\images com 158 imagens válidas
📚 Dataset inicializado: C:\\Users\\Jorge Cunha\\Desktop\\Python\\AI_Detection_Cars\\AAU2\\DetectObjects_JC\\dataset\\test\\images com 167 imagens válidas
🔨 Construindo modelo...
✅ Modelo pronto para treino
🚀 Iniciando treinamento...
📈 Epoch 1, Batch 1/315, Batch loss: 26.7795
📈 Epoch 1, Batch 2/315, Batch loss: 16.6005
📈 Epoch 1, Batch 3/315, Batch loss: 4.9323
📈 Epoch 1, Batch 4/315, Batch loss: 25.9152
📈 Epoch 1, Batch 5/315, Batch loss: 25.5117
📈 Epoch 1, Batch 6/315, Batch loss: 11.3365
📈 Epoch 1, Batch 7/315, Batch loss: 3.4450
📈 Epoch 1, Batch 8/315, Ba

In [2]:
import torch
from sklearn.metrics import precision_score, recall_score, f1_score
from torchvision import transforms as T
from torch.utils.data import DataLoader
import yaml
from PIL import Image
import os
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

# ⚙️ Dispositivo
device = torch.device("cpu")
print(f"🖥️ Usando dispositivo: {device}")

# 📂 Dataset
with open("dataset.yaml", "r") as file:
    dataset_config = yaml.safe_load(file)

val_dir = dataset_config["val"]
nc = dataset_config["nc"]
class_names = dataset_config["names"]

def load_yolo_labels(label_path, img_width, img_height):
    boxes, labels = [], []
    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, xc, yc, w, h = map(float, line.strip().split())
            x_min = (xc - w / 2) * img_width
            y_min = (yc - h / 2) * img_height
            x_max = (xc + w / 2) * img_width
            y_max = (yc + h / 2) * img_height
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(int(cls))
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

class YoloDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = img_dir.replace("images", "labels")
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(('jpg', 'png', 'jpeg'))]
        self.transform = transform

        self.imgs = [img for img in self.imgs if os.path.exists(
            os.path.join(self.label_dir, os.path.splitext(img)[0] + ".txt"))]

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        label_path = os.path.join(self.label_dir, os.path.splitext(self.imgs[idx])[0] + ".txt")

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = load_yolo_labels(label_path, w, h)
        target = {"boxes": boxes, "labels": labels}

        if self.transform:
            img = self.transform(img)

        return img, target

# 🔁 Dataloader
transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor()
])
val_dataset = YoloDataset(val_dir, transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# 🧠 Residual-like Backbone usado no treino
class ResidualLikeCNNBackbone(nn.Module):
    def __init__(self):
        super(ResidualLikeCNNBackbone, self).__init__()

        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels)
            )

        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.block1 = conv_block(32, 64)
        self.shortcut1 = nn.Conv2d(32, 64, kernel_size=1)

        self.block2 = conv_block(64, 128)
        self.shortcut2 = nn.Conv2d(64, 128, kernel_size=1)

        self.pool = nn.MaxPool2d(2)
        self.out_channels = 128

    def forward(self, x):
        x = self.layer1(x)
        res1 = self.block1(x) + self.shortcut1(x)
        res1 = F.relu(res1)
        res1 = self.pool(res1)

        res2 = self.block2(res1) + self.shortcut2(res1)
        res2 = F.relu(res2)
        res2 = self.pool(res2)

        return OrderedDict([("0", res2)])

# 📥 Carregar o modelo com backbone correto
backbone = ResidualLikeCNNBackbone()
anchor_gen = AnchorGenerator(sizes=((32, 64, 128, 256, 512),), aspect_ratios=((0.5, 1.0, 2.0),))
roi_pool = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)
model = FasterRCNN(backbone, num_classes=nc + 1, rpn_anchor_generator=anchor_gen, box_roi_pool=roi_pool)
model.load_state_dict(torch.load("models/ResidualLikeCNNBackbone_yolo_label.pth", map_location=device))
model.to(device)
model.eval()

# 📊 Avaliação
def evaluate_model(model, dataloader, threshold=0.0):
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, targets in dataloader:
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for t, p in zip(targets, preds):
                gt = t["labels"].cpu().numpy()
                scores = p["scores"].cpu().numpy()
                labels = p["labels"].cpu().numpy()
                pred_filtered = labels[scores > threshold]

                all_classes = set(gt.tolist() + pred_filtered.tolist())
                for cls in all_classes:
                    y_true.append(1 if cls in gt else 0)
                    y_pred.append(1 if cls in pred_filtered else 0)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n📌 Avaliação com limiar > {threshold}")
    print(f"🎯 Precisão: {precision:.3f}")
    print(f"🎯 Revocação: {recall:.3f}")
    print(f"🎯 F1-score: {f1:.3f}")

# 🚀 Executar
evaluate_model(model, val_loader, threshold=0.0)


🖥️ Usando dispositivo: cpu


C:\Users\Jorge Cunha\AppData\Local\Temp\ipykernel_15004\1827343492.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("models/faster_rcnn


📌 Avaliação com limiar > 0.0
🎯 Precisão: 0.380
🎯 Revocação: 0.085
🎯 F1-score: 0.139
